In [1]:
# TODO: Import necessary libraries
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
# TODO: Set pandas display options
# - display.max_columns to None
# - display.max_rows to 100
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# TODO: Print success message
print("Environment setup complete.")

Environment setup complete.


In [2]:
# TODO: Load cleaned data from Phase 1
# - Read 'train_cleaned.csv' using pd.read_csv()
# - Store in variable 'df'
df = pd.read_csv('../data/satilir_properties_cleaned.csv')
# TODO: Print dataset information
# - Dataset shape
# - Total missing values
# - Display first few rows with .head()
print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
df.head()

Dataset Shape: (4416, 25)
Total Missing Values: 0


,price,rooms,area_m2,land_area_sot,floor,has_document,address,avtodayanacaq,balkon,duzelme,esyali,hovuz,internet,isiq,kabel_tv,kombi,kondisioner,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirsiz
0,200000,3.0,65.0,0.0,5.0,No,"Bakı, Nərimanov, metro 28 May",Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,No,No
1,210000,2.0,54.0,0.0,12.0,No,"Bakı, Nərimanov, metro Nəriman Nərimanov",No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
2,149000,4.0,100.0,0.0,2.0,Yes,"Bakı, Binəqədi, Biləcəri qəs.",No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No
3,45000,1.0,44.0,0.0,1.0,Yes,Xırdalan,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
4,109000,3.0,69.0,0.0,13.0,No,Xırdalan,Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,No,Yes,No,No,Yes,Yes,Yes,No,No


In [3]:
# Detect Yes/No columns (encoding will be handled inside the preprocessing pipeline later)
yes_no_cols = []
for col in df.columns:
    non_null = df[col].dropna().astype(str).str.strip().str.lower()
    if len(non_null) and set(non_null.unique()).issubset({"yes", "no"}):
        yes_no_cols.append(col)

print("Detected Yes/No columns:", yes_no_cols)

Detected Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirsiz']


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4416 entries, 0 to 4415
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   price                     4416 non-null   int64  
 1   rooms                     4416 non-null   float64
 2   area_m2                   4416 non-null   float64
 3   land_area_sot             4416 non-null   float64
 4   floor                     4416 non-null   float64
 5   has_document              4416 non-null   object 
 6   address                   4416 non-null   object 
 7   avtodayanacaq             4416 non-null   object 
 8   balkon                    4416 non-null   object 
 9   duzelme                   4416 non-null   object 
 10  esyali                    4416 non-null   object 
 11  hovuz                     4416 non-null   object 
 12  internet                  4416 non-null   object 
 13  isiq                      4416 non-null   object 
 14  kabel_tv

In [5]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Count of Numerical Columns:", len(num_cols))
print("Numerical Columns:", num_cols)

Count of Numerical Columns: 5
Numerical Columns: ['price', 'rooms', 'area_m2', 'land_area_sot', 'floor']


In [6]:
df["address"].unique()

array(['Bakı, Nərimanov, metro 28 May',
       'Bakı, Nərimanov, metro Nəriman Nərimanov',
       'Bakı, Binəqədi, Biləcəri qəs.', 'Xırdalan', 'Bakı, Nəsimi',
       'Bakı, Nizami, metro Qara Qarayev',
       'Bakı, Xətai, Əhmədli, metro Həzi Aslanov',
       'Bakı, Suraxanı, Yeni Günəşli qəs.',
       'Bakı, Yasamal, metro İnşaatçılar', 'Bakı, Abşeron, Masazır',
       'Bakı, Binəqədi, Biləcəri qəs., metro Avtovağzal',
       'Bakı, Xəzər, Buzovna',
       'Bakı, Nizami, 8-ci kilometr, metro Neftçilər',
       'Bakı, Xətai, Əhmədli, metro Əhmədli',
       'Bakı, Yasamal, Yeni Yasamal qəs., metro İnşaatçılar',
       'Bakı, Abşeron, Qobu',
       'Bakı, Yasamal, Yasamal qəs., metro İnşaatçılar',
       'Bakı, Binəqədi, metro Azadlıq',
       'Bakı, Yasamal, Yeni Yasamal qəs., metro İçərişəhər',
       'Bakı, Binəqədi, 9-cu mikrorayon', 'Bakı, Sabunçu, metro Koroğlu',
       'Bakı, Nizami, metro Neftçilər', 'Bakı, Yasamal, metro 20 Yanvar',
       'Bakı, Xətai, H.Aslanov qəs.',
       '

In [7]:
address_parts = (
    df['address']
    .fillna('')
    .str.split(',', expand=True)
    .apply(lambda col: col.str.strip())
    .replace('', pd.NA)
)
address_parts.columns = [f'address_part_{i+1}' for i in range(address_parts.shape[1])]

for col in address_parts.columns:
    df[col] = address_parts[col]

print('Created address part columns:', address_parts.columns.tolist())
df[address_parts.columns].head()


Created address part columns: ['address_part_1', 'address_part_2', 'address_part_3', 'address_part_4']


,address_part_1,address_part_2,address_part_3,address_part_4
0,Bakı,Nərimanov,metro 28 May,None
1,Bakı,Nərimanov,metro Nəriman Nərimanov,None
2,Bakı,Binəqədi,Biləcəri qəs.,None
3,Xırdalan,None,None,None
4,Xırdalan,None,None,None


In [8]:
df.drop(columns=["address"], inplace=True)

In [9]:
df[address_parts.columns].isnull().sum()

address_part_1       0
address_part_2     632
address_part_3     959
address_part_4    3372
dtype: int64

In [10]:
df[address_parts.columns].value_counts(dropna=False)

address_part_1  address_part_2  address_part_3           address_part_4       
Bakı            Abşeron         Masazır                  NaN                      632
Xırdalan        NaN             NaN                      NaN                      606
Bakı            Yasamal         Yeni Yasamal qəs.        metro İnşaatçılar        192
                Suraxanı        Yeni Günəşli qəs.        NaN                      162
                Nərimanov       metro Nəriman Nərimanov  NaN                      154
                                                                                 ... 
                Binəqədi        9-cu mikrorayon          metro Memar Əcəmi - 2      1
                Yasamal         Yeni Yasamal qəs.        metro İçərişəhər           1
                Nəsimi          3-cü mikrorayon          metro 20 Yanvar            1
                Xətai           H.Aslanov qəs.           metro Xətai                1
                Abşeron         Ceyranbatan qəs.         NaN 

In [11]:
df['address_part_1'].unique()

array(['Bakı', 'Xırdalan', 'Lənkəran', 'Sumqayıt', 'Qəbələ', 'Siyazən'],
      dtype=object)

In [12]:
df['address_part_2'].unique()

array(['Nərimanov', 'Binəqədi', None, 'Nəsimi', 'Nizami', 'Xətai',
       'Suraxanı', 'Yasamal', 'Abşeron', 'Xəzər', 'Sabunçu', 'Səbail',
       '10-cu mikrorayon', '1-ci mikrorayon', 'St.Sumqayıt',
       '18-ci mikrorayon', '44-cü məhəllə', 'Qaradağ', 'Sumqayıt Bulvarı',
       '14-cü məhəllə', '9-cu mikrorayon', '6-ci mikrorayon',
       '72-ci məhəllə', '17-ci mikrorayon', '16-ci mikrorayon',
       '8-ci mikrorayon', '12-ci mikrorayon', '13-cü mikrorayon',
       '5-ci mikrorayon', '1-ci məhəllə', 'metro Memar Əcəmi',
       '36-ci məhəllə', '2-ci mikrorayon', '3-ci məhəllə',
       '15-ci məhəllə', '40-ci məhəllə'], dtype=object)

In [13]:
print("Before deleting rows, dataset shape:", df.shape)
df = df[~((df["address_part_1"] == "Bakı") & (df["address_part_2"].isnull()))]
print("After deleting rows, dataset shape:", df.shape)

Before deleting rows, dataset shape: (4416, 28)
After deleting rows, dataset shape: (4416, 28)


In [14]:
df[(df["address_part_1"] != "Bakı") & (df["address_part_2"].isnull())]

,price,rooms,area_m2,land_area_sot,floor,has_document,avtodayanacaq,balkon,duzelme,esyali,hovuz,internet,isiq,kabel_tv,kombi,kondisioner,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirsiz,address_part_1,address_part_2,address_part_3,address_part_4
3,45000,1.0,44.0,0.0,1.0,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4,109000,3.0,69.0,0.0,13.0,No,Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,No,Yes,No,No,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
5,70500,2.0,73.0,0.0,4.0,No,Yes,No,No,No,No,No,Yes,Yes,Yes,No,No,No,No,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
7,135000,3.0,100.0,0.0,9.0,No,Yes,Yes,No,Yes,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
8,54000,1.0,31.0,0.0,5.0,No,Yes,No,No,No,No,Yes,Yes,Yes,Yes,No,Yes,No,No,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4397,92000,2.0,65.0,0.0,2.0,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4406,92000,3.0,66.0,0.0,1.0,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4407,120000,4.0,96.0,0.0,1.0,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4408,230000,4.0,164.0,0.0,11.0,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None


In [15]:
# df[(df["cities"] != "Bakı") & (df["address_part_2"].isnull()), "address_part_2"]
df.loc[(df["address_part_2"] != "Bakı") & (df["address_part_2"].isnull()), "address_part_2"] = df.loc[(df["address_part_1"] != "Bakı") & (df["address_part_2"].isnull()), "address_part_1"]

In [16]:
df.loc[(df["address_part_1"] != "Bakı"), ["address_part_1" , "address_part_2"]]

,address_part_1,address_part_2
3,Xırdalan,Xırdalan
4,Xırdalan,Xırdalan
5,Xırdalan,Xırdalan
7,Xırdalan,Xırdalan
8,Xırdalan,Xırdalan
...,...,...
4397,Xırdalan,Xırdalan
4406,Xırdalan,Xırdalan
4407,Xırdalan,Xırdalan
4408,Xırdalan,Xırdalan


In [17]:
df[address_parts.columns].isnull().sum()

address_part_1       0
address_part_2       0
address_part_3     959
address_part_4    3372
dtype: int64

In [18]:
df['address_part_3'].unique()

array(['metro 28 May', 'metro Nəriman Nərimanov', 'Biləcəri qəs.', None,
       'metro Qara Qarayev', 'Əhmədli', 'Yeni Günəşli qəs.',
       'metro İnşaatçılar', 'Masazır', 'Buzovna', '8-ci kilometr',
       'Yeni Yasamal qəs.', 'Qobu', 'Yasamal qəs.', 'metro Azadlıq',
       '9-cu mikrorayon', 'metro Koroğlu', 'metro Neftçilər',
       'metro 20 Yanvar', 'H.Aslanov qəs.', 'metro Memar Əcəmi',
       'metro İçərişəhər', '8-ci mikrorayon', 'metro Həzi Aslanov',
       'metro Xalqlar dostluğu', 'metro Xətai',
       'metro Elmlər akademiyası', '7-ci mikrorayon', 'metro 8 Noyabr',
       'Saray', 'metro Gənclik', 'Bakıxanov qəs.', 'metro Nizami',
       'Badamdar qəs.', 'Qara şəhər', 'metro Əhmədli', 'metro Sahil',
       'Nardaran qəs.', '3-cü mikrorayon', 'Köhnə Günəşli qəs.',
       'metro Nəsimi', 'Əmircan qəs.', 'Lökbatan qəs.', 'Qaraçuxur qəs.',
       'Hövsan qəs.', 'Ramana qəs.', 'Ağ şəhər', '6-cı mikrorayon',
       'Bayıl qəs.', 'Binəqədi qəs.', 'Montin qəs.', 'Zığ qəs.',
      

In [19]:
df['address_part_4'].unique()

array([None, 'metro Həzi Aslanov', 'metro Avtovağzal', 'metro Neftçilər',
       'metro Əhmədli', 'metro İnşaatçılar', 'metro İçərişəhər',
       'metro Nəsimi', 'metro Azadlıq', 'metro Elmlər akademiyası',
       'metro 20 Yanvar', 'metro Dərnəgül', 'metro Xalqlar dostluğu',
       'metro Gənclik', 'metro Qara Qarayev', 'metro Koroğlu',
       'metro Xətai', 'metro Memar Əcəmi', 'metro Nəriman Nərimanov',
       'metro Memar Əcəmi - 2', 'metro 8 Noyabr', 'metro 28 May',
       'metro Nizami', 'metro Xocaəsən', 'metro Sahil'], dtype=object)

In [20]:
df.drop(columns=["address_part_3", "address_part_4"], inplace=True)

In [21]:
df.shape

(4416, 26)

In [22]:
df.rename(columns={"address_part_1": "cities"}, inplace=True)
df.rename(columns={"address_part_2": "raion"}, inplace=True)

In [23]:
print("Number of unique cities:", df['cities'].nunique())
print(f"Cardinality of cities: {(df['cities'].nunique()/df.shape[0]) * 100:.2f}%")

Number of unique cities: 6
Cardinality of cities: 0.14%


In [24]:
print("Number of unique cities:", df['raion'].nunique())
print(f"Cardinality of cities: {(df['raion'].nunique()/df.shape[0]) * 100:.2f}%")

Number of unique cities: 40
Cardinality of cities: 0.91%


In [25]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

class YesNoBinaryEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.feature_names_in_ = X_df.columns.to_numpy()
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.feature_names_in_)
        out = pd.DataFrame(index=X_df.index)
        for col in self.feature_names_in_:
            out[col] = (
                X_df[col]
                .astype("string")
                .str.strip()
                .str.lower()
                .map({"yes": 1, "no": 0})
                .astype(float)
            )
        return out.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return np.asarray(input_features, dtype=object)

class MultiColumnFrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps_ = {}
        self.columns_ = []

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = X_df.columns.tolist()
        self.feature_names_in_ = np.asarray(self.columns_, dtype=object)
        self.freq_maps_ = {
            col: X_df[col].astype("string").value_counts(normalize=True, dropna=False).to_dict()
            for col in self.columns_
        }
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.columns_)
        out = pd.DataFrame(index=X_df.index)
        for col in self.columns_:
            out[f"{col}_freq"] = (
                X_df[col].astype("string").map(self.freq_maps_[col]).fillna(0.0).astype(float)
            )
        return out.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)
        return np.asarray([f"{col}_freq" for col in input_features], dtype=object)

categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in yes_no_cols]

high_card_cols = [c for c in categorical_cols if df[c].nunique(dropna=False) > 20]
low_card_cols = [c for c in categorical_cols if c not in high_card_cols]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("Yes/No columns:", yes_no_cols)
print("Low-cardinality categorical columns:", low_card_cols)
print("High-cardinality categorical columns:", high_card_cols)
print("Numeric columns:", len(numeric_cols))

preprocessor = ColumnTransformer(
    transformers=[
        (
            "yes_no_binary",
            Pipeline([("yes_no", YesNoBinaryEncoder())]),
            yes_no_cols
        ),
        (
            "cat_ohe",
            Pipeline([("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            low_card_cols
        ),
        (
            "cat_freq",
            Pipeline([("freq", MultiColumnFrequencyEncoder())]),
            high_card_cols
        ),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
    sparse_threshold=0
)

full_pipeline = Pipeline([("preprocessor", preprocessor)])

X_processed = full_pipeline.fit_transform(df)
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
processed_df = pd.DataFrame(X_processed, columns=feature_names, index=df.index)

print("Processed feature matrix shape:", processed_df.shape)
processed_df.head()

Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirsiz']
Low-cardinality categorical columns: ['cities']
High-cardinality categorical columns: ['raion']
Numeric columns: 5
Processed feature matrix shape: (4416, 31)


,yes_no_binary__has_document,yes_no_binary__avtodayanacaq,yes_no_binary__balkon,yes_no_binary__duzelme,yes_no_binary__esyali,yes_no_binary__hovuz,yes_no_binary__internet,yes_no_binary__isiq,yes_no_binary__kabel_tv,yes_no_binary__kombi,yes_no_binary__kondisioner,yes_no_binary__lift,yes_no_binary__merkezi_qizdirici_sistem,yes_no_binary__metbex_mebeli,yes_no_binary__pvc_pencere,yes_no_binary__qaz,yes_no_binary__su,yes_no_binary__telefon,yes_no_binary__temirsiz,cat_ohe__cities_Bakı,cat_ohe__cities_Lənkəran,cat_ohe__cities_Qəbələ,cat_ohe__cities_Siyazən,cat_ohe__cities_Sumqayıt,cat_ohe__cities_Xırdalan,cat_freq__raion_freq,num__price,num__rooms,num__area_m2,num__land_area_sot,num__floor
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.076087,200000.0,3.0,65.0,0.0,5.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.076087,210000.0,2.0,54.0,0.0,12.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.066350,149000.0,4.0,100.0,0.0,2.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.137228,45000.0,1.0,44.0,0.0,1.0
4,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.137228,109000.0,3.0,69.0,0.0,13.0


In [26]:
# TODO: Save the cleaned dataset to CSV
processed_df.to_csv('../data/satilir_properties_preprocessed.csv', index=False)
# - Print confirmation message
print("Preprocessed dataset saved to 'data/satilir_properties_preprocessed.csv'")

Preprocessed dataset saved to 'data/satilir_properties_preprocessed.csv'
